# Production notebook building masking like an agent would do it

The goal of this notebook is to go through what an agent should do when producing a mask for a given run, and use this exploration to identify simplifications and clarifications necessary in the codebase.

**Plan** (input = RUN):
1. Explore content of the run + associated calibration data 
```python
list_contents(RUN)
# Should list number of shots, different properties, number of shots available per property, calibration data available
# Geometry of the detector etc ...
```
2. Shot selection : based on content, decide on several shot selections and run them
```python
select_shots(...) # Selection 1
select_shots(...) # Selection 2
select_shots(...) # Selection 3
display_shots() # Displays shots together for visual analysis and refine/discard some if necessary
```
3. Masking : based on observed shots, decide what masking tools to use and apply them (always apply geometry + calib first)
```python
detector(...) # Masking : includes stats, regularization etc ...
display_masks() # Display each mask obtained and refine parameters or tools if necessary
```
4. Validation : based on obtained masks, apply validation methods to refine previous behavior if necessary
```python
raise(NotImplemented)
```

In [ ]:
from automask.utils import configure_psana_environment

PSANA_ENVIRONMENT = configure_psana_environment()

## 0. Experiment content

This first stage only identifies the experiment and resolves the files relevant to the requested run and detector. It does not open or analyze any data. In practice, the agent should recover this context; it is explicit here so the notebook remains reproducible.

Inputs to clarify before starting:
1. Experiment name
2. Run number
3. Detector alias, psana source, and calibration type
4. Data paths: find the run's `.xtc` streams and the detector calibration files applicable to that run.

Expected output:
```markdown
Experiment: ""
Run: ""
Detector: ""
Relevant files:
- XTC files (Markdown table)
- Applicable detector calibration files (Markdown table)
```

In [ ]:
# Hardcoded information for this run

from automask.io.read_xtc import JUNGFRAU_NAME

EXPERIMENT_NAME = 'xppl1016922'
RUN = 475
DETECTOR_NAME = JUNGFRAU_NAME
DETECTOR_SOURCE = 'XppEndstation.0:Jungfrau.0'
DETECTOR_CALIB_TYPE = 'Jungfrau::CalibV1'

In [ ]:
from automask.utils import (
    list_experiment_content,
    profile_run_values,
    print_detector_geometry,
)

In [ ]:
experiment_content = list_experiment_content(
    EXPERIMENT_NAME, RUN, DETECTOR_NAME, DETECTOR_SOURCE, DETECTOR_CALIB_TYPE
)
xtc = experiment_content['xtc']

## 1. Analysis of the listed files

This stage is reached once the available files and XTC keys have been listed. It profiles their values, reads detector geometry, and analyzes the relevant calibration constants.

At this stage, available and confirmed information should be:
1. RUN
2. Experiment name
3. Detector alias 
4. XTC and calibration files

### Profile XTC and EPICS values and geometry

The run profiler discovers payloads directly from `event.keys()` and extracts the known numeric fields from every decoded event. After each event it also records the current value of every process variable in `data_source.env().epicsStore()`, including both its alias and underlying PV name. It does not require experiment-specific payload or value definitions.

```python
run_profile = profile_run_values(RUN)
print_detector_geometry(RUN)
```

In [ ]:
run_profile = profile_run_values(RUN)
print_detector_geometry(RUN)

> [!warning] 
>
> Some key information is sometimes not available from the configuration files. Examples include sample-detector distance, ?? (#TODO include others). The agent should be able to identify this and either 
> 1. Retrieve if from log files or other sources
> 2. Ask the user for this information
> 3. Proceed to calibrate the distance itself from the data (if possible)

## 2. Shot selection

At this stage, the agent should have a good understanding of the available shots and their properties. It should build a few shot selections based on the available information and the goals of the analysis. It should follow the guidelines provided in the skill. 

Later, the agent will then analyze the selected shots and refine or discard some of them if necessary.

### Hardcoded shot selection

In [ ]:
from automask.shot_selection import ShotSelection

LIT_SELECTION = ShotSelection(
    beam='on',
    cc='open',
    vcc='closed',
    n_shots=800,
    filter_low=0.03,
    filter_high=0.03,
    intensity='sample_diode',
    normalization='none',
)

DARK_SELECTION = ShotSelection(
    beam='off',
    cc='any',
    vcc='any',
    n_shots=None,
    filter_low=0.0,
    filter_high=0.0,
    intensity='sample_diode',
    normalization='none',
)

In [ ]:
from automask.features import FeatureSpec, FeatureStore
from automask.viz import show

features = {
    'Lit calibrated mean': FeatureSpec('lit_mean', 'mean', LIT_SELECTION),
    'Beam-off calibrated median': FeatureSpec('dark_median', 'median', DARK_SELECTION),
}
store = FeatureStore(run_profile=run_profile)
selected_images = {name: store.get(RUN, feature) for name, feature in features.items()}

for name, image in selected_images.items():
    show(image, title=f'Run {RUN:04d}: {name.lower()}')

## 3. Masking

### Hardcoded masking techniques and parameters.

In [ ]:
from types import SimpleNamespace
from automask.dataset import load_mask
from automask.masking import Detector, Pipeline
from automask.stats.blackhat import BlackhatParams
from automask.regularization.tv import TVParams
from automask.regularization.pad import PadParams
from automask.viz import show_mask

MASKING_PIPELINE = Pipeline(
    detectors=[Detector(
        stat='blackhat',
        stat_params=BlackhatParams(radius=5, k=6.0, mode='high'),
        field_reg='tv',
        field_reg_params=TVParams(weight=1.0),
        mask_reg='pad',
        mask_reg_params=PadParams(pad=2),
    )],
    floor_stats=["geometry", "calib"],
    combiner='union',
)

image = selected_images['Lit calibrated mean']
calib = load_mask(f'statusMask_run{RUN:04d}')
sample = SimpleNamespace(sumimg=image, umean=image, real=image != 0, calib=calib)
computed_mask = MASKING_PIPELINE.run(sample)
show_mask(computed_mask, title=f'Run {RUN:04d}: {computed_mask.mean():.2%} masked')